# 36. scikit-image Analysis — `scikit-image`
**Python Libraries Lab · Phase 2 · Computer Vision**

_Scientific image analysis: segment, measure and count._

✅ **Runs in Colab** as-is.

scikit image is the scientific side of image processing. Where Pillow edits photos, scikit image measures them. It can separate objects from the background, label each object, and give you measurements like area, size and position. Biologists count cells with it, engineers check parts, and shops can count items on a shelf.

**In this notebook:** You will count and measure the coins in a sample photo, clean up the mask with morphology, compare edge filters, and resize and rotate images while keeping their quality.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q scikit-image
import importlib.metadata as md
for pkg in ['scikit-image']:
    print(pkg, md.version(pkg))

## Step 1 · Load the coins photo
A sample image bundled with scikit-image.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from skimage import data, filters, measure, morphology, segmentation, color, transform, exposure

coins = data.coins()
print(coins.shape, coins.dtype, coins.min(), coins.max())
plt.imshow(coins, cmap="gray"); plt.axis("off"); plt.title("Coins"); plt.show()

## Step 2 · Threshold with Otsu
Otsu chooses the brightness cut-off automatically.

In [ ]:
t = filters.threshold_otsu(coins)
mask = coins > t
cleaned = morphology.remove_small_objects(morphology.closing(mask, morphology.disk(3)), 100)
cleaned = segmentation.clear_border(cleaned)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(mask, cmap="gray"); axes[0].set_title(f"Otsu threshold = {t:.0f}")
axes[1].imshow(cleaned, cmap="gray"); axes[1].set_title("Cleaned with morphology")
for a in axes: a.axis("off")
plt.show()

## Step 3 · Label and count
Each connected blob gets its own number and colour.

In [ ]:
labels = measure.label(cleaned)
props = measure.regionprops(labels)
print("Coins found:", labels.max())
plt.imshow(color.label2rgb(labels, image=coins, bg_label=0)); plt.axis("off"); plt.title("Each coin labelled"); plt.show()

## Step 4 · Measure each object
`regionprops_table` gives a table you can load into pandas.

In [ ]:
import pandas as pd
table = pd.DataFrame(measure.regionprops_table(labels, properties=["label", "area", "equivalent_diameter_area", "centroid"]))
table["size"] = pd.cut(table["equivalent_diameter_area"], bins=3, labels=["small", "medium", "large"])
print(table.round(1).head(10))
print("\nCoins by size:", table["size"].value_counts().to_dict())

## Step 5 · Edges, contrast and transforms
Compare edge filters, stretch contrast, rescale and rotate.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
views = {
    "Sobel edges": filters.sobel(coins),
    "Scharr edges": filters.scharr(coins),
    "Equalised": exposure.equalize_adapthist(coins),
    "Half size": transform.rescale(coins, 0.5, anti_aliasing=True),
    "Rotated 30°": transform.rotate(coins, 30, resize=True),
}
for ax, (name, im) in zip(axes, views.items()):
    ax.imshow(im, cmap="gray"); ax.set_title(name); ax.axis("off")
plt.show()

## ✍️ Your turn
Store the **area of the largest coin** (from `props`) in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    exp = max(p.area for p in props)
    return answer == exp, f"Expected {exp}, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = max(p.area for p in props)

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Threshold (Otsu)**
- **Segmentation**
- **Label**
- **regionprops**
- **Morphology**
- **Filters**

**Next:** 37. Django Web Framework (`django`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/37_django.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install scikit-image
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.